**Introduction et contexte :**

This notebook covers the **ridge** *regression model* in the context of our machine learning project.

Rappel de la formule venant du TP7:


 In the regression setting, given a training dataset $\mathcal{D} = \{(x^{(i)},y^{(i)})\}_{i=1}^n$, recall from the course that ridge regression aims to find the coefficients ${\beta}^\text{R}$ that mimimize the following optimization problem:
$${\beta}^\text{R} = \underset{{\beta}}{\text{argmin}} \sum_{i=1}^n \left(y^{(i)} - \beta_0 - \sum_{j=1}^p \beta_j x^{(i)}_{j}\right)^2 + \lambda \sum_{j=1}^p \beta_j^2,$$
where $\lambda$ is a hyperparameter controlling the amount of shrinkage applied to the coefficients $\beta_j$.

Aspects à prendre en compte :


*   Les estimateurs peuvent changer si les prédicteurs sont multiplié par une constante, dû à l'addition des coefficient au carré dans la formule. (CH5 sld 49/64)
*   Si on ne prend pas en compte le scaling, le prédicteur avec la plus grosse échelle peut dominer les estimateur de la **ridge** regression (CH5 sld 50/64)



**Librairies nécessaires :**

In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    mean_squared_error,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, train_test_split
import sys
from pathlib import Path
import pandas as pd

**Récolte des données :**

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "ipynb" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import data_loader

features = data_loader.load_all_data()
results_train = pd.read_csv(DATA_DIR / "results_train.csv")

left_id_col = "CommuneID"
right_id_col = "Gemeinde-Nummer"
target_col = "Ja in Prozent"

# On renomme la vraie cible pour éviter les conflits de noms après merge
target_df = results_train[[right_id_col, target_col]].copy()
target_df = target_df.rename(columns={target_col: "target"})

train_df = features.merge(
    target_df,
    left_on=left_id_col,
    right_on=right_id_col,
    how="inner"
)

# y = variable à prédire
y = train_df["target"]

# X = variables explicatives
X = train_df.drop(columns=["target", right_id_col])

# Garder uniquement les colonnes numériques pour Ridge
X = X.select_dtypes(include="number")

# Retirer les identifiants
cols_to_remove = [col for col in ["CommuneID", "CantonID"] if col in X.columns]
X = X.drop(columns=cols_to_remove)

print("features shape:", features.shape)
print("results_train shape:", results_train.shape)
print("train_df shape:", train_df.shape)
print("X shape:", X.shape)
print("y shape:", y.shape)

display(X.head())
display(y.head())

features shape: (2325, 74)
results_train shape: (1559, 13)
train_df shape: (1559, 76)
X shape: (1559, 69)
y shape: (1559,)


,Stimmberechtigte,eingelegte Stimmzettel,Stimmbeteiligung,leere Stimmzettel,ungültige Stimmzettel,gültige Stimmen,Ja-Stimmen,Nein-Stimmen,Ja in Prozent,stpf,...,SP,SVP,EVP/CSP,GLP,BDP,PdA/Sol.,GPS,Small right-wing parties,lat,lon
0,1402.0,662.0,47.218260,5.0,0.0,657.0,206.0,451.0,31.354642,900.0,...,18.645940,30.929249,3.467063,8.435250,2.617442,0.167638,7.075094,4.888178,47.267500,8.485278
1,7186.0,2877.0,40.036181,49.0,0.0,2828.0,695.0,2133.0,24.575672,5385.0,...,19.080314,33.785785,5.464827,7.357860,4.164299,0.190049,6.211047,1.768197,47.281670,8.450230
2,3656.0,1690.0,46.225383,25.0,1.0,1664.0,404.0,1260.0,24.278846,2457.0,...,20.403265,29.100156,3.143003,11.862398,3.803108,0.112518,6.661066,1.915807,47.315556,8.469167
3,2505.0,1140.0,45.508982,14.0,0.0,1126.0,353.0,773.0,31.349911,1613.0,...,19.393305,34.937369,2.569875,8.748273,4.656087,0.193911,8.021665,1.825436,47.244722,8.534444
4,2515.0,1180.0,46.918489,12.0,0.0,1168.0,312.0,856.0,26.712329,1639.0,...,22.478008,30.114599,3.589299,9.625938,3.768864,0.227988,6.466387,1.840045,47.298611,8.449444


0    49.347258
1    45.572227
2    43.608653
3    49.331457
4    46.065129
Name: target, dtype: float64

**Division des données en 70% train et 30% test**

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.7, test_size=0.3, random_state=0
)

**Selection de la range de** $\lambda$

Pour l'instant la range de $\lambda$ va de 0.1 à 10 avec des steps de 0.1. Si nous trouvons un meilleur intervalle n'hésitons pas à le changer.
(les valeurs viennent du TP7)

In [ ]:
alpha = np.arrange(0.1,10,0.1) #Valeur que peuvent prendre les lambda

**Trouver le meilleur** $λ$

In [ ]:
model = Ridge(fit_intercept=True) #Selection du modèle

grid = {'alpha': alpha} #Créer une grille avec les valeurs de alpha

reg_ridge = GridSearchCV(model,grid, scoring='neg_root_mean_squared_error') #Scoring demandé dans le PDF du projet ?

reg_ridge.fit(X_train,y_train) #Fit le modèle avec les données d'entraînement

**Refit avec le meilleur** $λ$

In [ ]:
best_lambda = reg_ridge.best_params_['alpha'] #Selectionner le meilleur lambda dans les paramètres du moèle fit

best_ridge = Ridge(alpha=best_lambda, fit_intercept=True) #Créer un modèle avec comme lambda la meilleure valeur trouvée plus haut

best_ridge.fit(X_train,y_train) #Refit les données avec le nouveau modèle.

**Analyse des résultats**